# CropCop Track-A — K2 Context Controller

Hard-bound to execution source `8904b100d223e4319776199c87ab397db23600ce`. T4x2 runs the fixed EfficientNet-B0/ConvNeXt-Tiny context pair with private-dataset continuation.


In [ ]:
PHASE = "secondary-scientific"
EXPECTED_KAGGLE_USERNAME = "sabahatabbas"
LANE = "K2"
SCIENCE_ENVELOPE = "context"
BACKFILL_PHASE = "principal-backfill-s2"
BACKFILL_SEED = "S2"

# After secondary science is terminal, PHASE may optionally become "principal-backfill-s2".


## Locked bootstrap

Do not edit this cell. Accepted execution is **Save Version → Save & Run All** only.


In [ ]:
import os, sys, time, shutil, subprocess, tempfile, stat, hashlib, json, csv
from pathlib import Path

AUTHORIZED_SOURCE_SHA = "8904b100d223e4319776199c87ab397db23600ce"
SOURCE_BRANCH = "je-tracka-secondary-core-20260910"
REPOSITORY_URL = "https://github.com/rana-m-ahmed/ResearchWork-CropCop.git"
PRINCIPAL_SOURCE_SHA = "f171309fc7e9dc22241ecc137ebbb8e4bcdc5433"
PRINCIPAL_G1_SEAL_SHA256 = "442d9e7708749efedcb82ef9f4fd131211549770eecad985177eaaf4117052cd"
MANIFEST_SHA256 = "bdb82211ccc2059153724eea178a1680893a6b38ecc243fae484baa91dbf68e2"
CLASS_MAP_SHA256 = "46f7811726c19c42bd7213b2d8178b19a5a182a1b763f60a94ee2c0e5f6688d2"

os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
os.environ["CROPCOP_NOTEBOOK_STARTED_MONOTONIC"] = repr(time.monotonic())
os.environ.setdefault("CROPCOP_NOTEBOOK_HARD_LIMIT_SECONDS", str(12 * 3600))
os.environ.setdefault("CROPCOP_NOTEBOOK_FINALIZATION_MARGIN_SECONDS", str(3600))

if sys.version.split()[0] != "3.12.13":
    raise RuntimeError(f"Python re-lock required: {sys.version.split()[0]}")

run_type = str(os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "")).strip()
if run_type != "Batch":
    raise RuntimeError(
        "Accepted execution requires Kaggle Save Version -> Save & Run All. "
        f"Observed KAGGLE_KERNEL_RUN_TYPE={run_type or '<missing>'}."
    )

def gpu_names():
    exe = shutil.which("nvidia-smi")
    if not exe:
        return []
    cp = subprocess.run(
        [exe, "--query-gpu=name", "--format=csv,noheader"],
        capture_output=True, text=True, check=False, timeout=30
    )
    return [x.strip() for x in cp.stdout.splitlines() if x.strip()] if cp.returncode == 0 else []

names = gpu_names()
if PHASE in {"dual-gpu-smoke", "secondary-g2", "secondary-scientific", "principal-backfill-s1", "principal-backfill-s2", "principal-backfill-s3"}:
    if len(names) != 2 or any(x not in {"Tesla T4", "NVIDIA T4"} for x in names):
        raise RuntimeError(f"{PHASE} requires exactly T4 x2; observed {names}")
elif PHASE in {"smoke-write", "smoke-restore"}:
    if not names:
        raise RuntimeError(f"{PHASE} requires a CUDA-capable Kaggle GPU")
elif PHASE == "secondary-g1":
    print(f"Secondary G1 CPU phase: visible GPUs={names}")

def get_secret(name):
    value = str(os.environ.get(name, "")).strip()
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        value = str(UserSecretsClient().get_secret(name) or "").strip()
    except Exception as exc:
        raise RuntimeError(f"Missing Kaggle Secret {name}") from exc
    if not value:
        raise RuntimeError(f"Missing Kaggle Secret {name}")
    return value

github_token = get_secret("CROPCOP_GITHUB_TOKEN")
os.environ["CROPCOP_GITHUB_TOKEN"] = github_token

KAGGLE_API_PHASES = {
    "secondary-g1", "secondary-g2", "secondary-scientific",
    "principal-backfill-s1", "principal-backfill-s2", "principal-backfill-s3"
}
if PHASE in KAGGLE_API_PHASES:
    kaggle_username = get_secret("KAGGLE_USERNAME")
    kaggle_key = get_secret("KAGGLE_KEY")
    if kaggle_username.casefold() != EXPECTED_KAGGLE_USERNAME.casefold():
        raise RuntimeError(
            f"Wrong Kaggle account for this phase: expected {EXPECTED_KAGGLE_USERNAME}, observed {kaggle_username}"
        )
    os.environ["KAGGLE_USERNAME"] = kaggle_username
    os.environ["KAGGLE_KEY"] = kaggle_key

os.environ["CROPCOP_SOURCE_GIT_COMMIT"] = AUTHORIZED_SOURCE_SHA
os.environ["CROPCOP_LANE"] = LANE
os.environ["CROPCOP_ROW_ID_COLUMN"] = "record_key"
os.environ["CROPCOP_PATH_COLUMN"] = "portable_relpath"
os.environ["CROPCOP_SPLIT_COLUMN"] = "split"
os.environ["CROPCOP_LABEL_COLUMN"] = "label"
os.environ["CROPCOP_TRAIN_SPLIT_VALUE"] = "train"
os.environ["CROPCOP_VAL_SPLIT_VALUE"] = "val"

repo = Path("/kaggle/working/cropcop-secondary-src").resolve()
if repo.exists():
    shutil.rmtree(repo)

with tempfile.TemporaryDirectory() as td:
    askpass = Path(td) / "askpass.py"
    askpass.write_text(
        "#!/usr/bin/env python3\n"
        "import os,sys\n"
        "p=(sys.argv[1] if len(sys.argv)>1 else '').lower()\n"
        "print('x-access-token' if 'username' in p else os.environ['CROPCOP_GITHUB_TOKEN'])\n"
    )
    askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)
    git_env = dict(os.environ)
    git_env["GIT_ASKPASS"] = str(askpass)
    git_env["GIT_TERMINAL_PROMPT"] = "0"
    git_env["GIT_ASKPASS_REQUIRE"] = "force"

    subprocess.run(
        ["git", "clone", "--no-checkout", "--filter=blob:none", REPOSITORY_URL, str(repo)],
        env=git_env, check=True
    )
    direct = subprocess.run(
        ["git", "-C", str(repo), "fetch", "--depth=1", "origin", AUTHORIZED_SOURCE_SHA],
        env=git_env, capture_output=True, text=True
    )
    if direct.returncode != 0:
        subprocess.run(
            ["git", "-C", str(repo), "fetch", "--depth=64", "origin", SOURCE_BRANCH],
            env=git_env, check=True
        )
    subprocess.run(
        ["git", "-C", str(repo), "checkout", "--detach", AUTHORIZED_SOURCE_SHA],
        env=git_env, check=True
    )

head = subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip()
if head != AUTHORIZED_SOURCE_SHA:
    raise RuntimeError(f"Exact source checkout failed: {head}")
dirty = subprocess.check_output(
    ["git", "-C", str(repo), "status", "--porcelain=v1", "--untracked-files=all"], text=True
).strip()
if dirty:
    raise RuntimeError("Source checkout is not clean")

lockfile = repo / "journal_extension/requirements-training.lock.txt"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--no-input", "-r", str(lockfile)],
    check=True
)

subprocess.run(
    [
        sys.executable,
        str(repo / "journal_extension/scripts/check_secondary_science_diff.py"),
        "--repo-root", str(repo),
        "--output", "/kaggle/working/secondary-science-diff-report.json",
    ],
    cwd=repo, check=True
)

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def json_candidates(filename):
    result = []
    for p in sorted(Path("/kaggle/input").rglob(filename)):
        if not p.is_file():
            continue
        try:
            payload = json.loads(p.read_text(encoding="utf-8"))
        except Exception:
            continue
        if isinstance(payload, dict):
            result.append((p.resolve(), payload))
    return result

def find_bound_json(filename, *, source_sha=None, status=None, extra=None, label=None):
    accepted = []
    for path, payload in json_candidates(filename):
        if source_sha is not None and payload.get("source_git_sha") != source_sha:
            continue
        if status is not None and payload.get("status") != status:
            continue
        if extra is not None and not extra(payload):
            continue
        accepted.append(path)
    if len(accepted) != 1:
        raise RuntimeError(
            f"Expected exactly one {label or filename} bound to required identity; "
            f"found {len(accepted)}: {accepted}"
        )
    return accepted[0]

def find_final_v1():
    manifests = [
        p.resolve() for p in Path("/kaggle/input").rglob("final_manifest.csv")
        if p.is_file() and sha256(p) == MANIFEST_SHA256
    ]
    candidates = []
    for manifest in manifests:
        class_map = manifest.parent / "class_to_idx.json"
        if not class_map.is_file() or sha256(class_map) != CLASS_MAP_SHA256:
            continue
        with manifest.open("r", encoding="utf-8", newline="") as fh:
            reader = csv.DictReader(fh)
            sample = []
            for row in reader:
                if row.get("split") in {"train", "val"} and row.get("portable_relpath"):
                    sample.append(row["portable_relpath"])
                    if len(sample) >= 24:
                        break
        base = manifest.parent.parent
        roots = [base, base.parent]
        scored = [(sum((root / rel).is_file() for rel in sample), root) for root in roots]
        scored.sort(key=lambda x: x[0], reverse=True)
        if sample and scored[0][0] == len(sample):
            candidates.append((manifest, class_map.resolve(), scored[0][1].resolve()))
    if len(candidates) != 1:
        raise RuntimeError(
            "Could not uniquely resolve frozen Final-V1 by exact manifest/class-map hashes. "
            f"Candidates={[(str(a), str(c), str(r)) for a,c,r in candidates]}"
        )
    return candidates[0]

print("Bootstrap PASS")
print("Exact execution source:", AUTHORIZED_SOURCE_SHA)
print("Phase:", PHASE)
print("Visible GPUs:", names)


## Phase execution


In [ ]:
from pathlib import Path
import subprocess, sys, os, json

if PHASE == "secondary-scientific":
    manifest, class_map, image_root = find_final_v1()
    sec_seal = find_bound_json(
        "SECONDARY_G1_MODEL_IDENTITY_SEAL.json",
        source_sha=AUTHORIZED_SOURCE_SHA,
        label="qualified Secondary-G1 seal",
    )
    g2_barrier = find_bound_json(
        "SECONDARY_G2_CALIBRATION_BARRIER.json",
        source_sha=AUTHORIZED_SOURCE_SHA,
        status="PASS",
        label="terminal Secondary-G2 barrier",
    )
    os.environ["CROPCOP_MANIFEST"] = str(manifest)
    os.environ["CROPCOP_CLASS_MAP"] = str(class_map)
    os.environ["CROPCOP_IMAGE_ROOT"] = str(image_root)
    os.environ["CROPCOP_SECONDARY_G1_INPUT_ROOT"] = str(sec_seal.parent)
    os.environ["CROPCOP_SECONDARY_G2_BARRIER"] = str(g2_barrier)
    os.environ["CROPCOP_SECONDARY_ENVELOPE"] = SCIENCE_ENVELOPE
    os.environ["CROPCOP_OUTPUT_ROOT"] = f"/kaggle/working/cropcop-secondary-{LANE.lower()}"
    os.environ["CROPCOP_DURABLE_STORE_KIND"] = "kaggle-dataset"
    os.environ["CROPCOP_DURABLE_LOCATOR_TEMPLATE"] = f"{EXPECTED_KAGGLE_USERNAME}/{{run_id_lower}}"
    os.environ["CROPCOP_SECONDARY_ALLOW_CREATE_PRIVATE_DATASETS"] = "1"
    subprocess.run([
        sys.executable, str(repo / "journal_extension/kaggle/run_secondary_envelope.py")
    ], cwd=repo, check=True)

    evidence = Path(os.environ["CROPCOP_OUTPUT_ROOT"]) / "SECONDARY_ENVELOPE_EVIDENCE.json"
    if not evidence.is_file():
        raise RuntimeError("Secondary envelope finished without evidence file")
    payload = json.loads(evidence.read_text())
    print("Envelope status:", payload.get("status"))
    for cid, row in payload.get("children", {}).items():
        print(cid, "execution=", row.get("execution_status"), "publication=", row.get("publication_status"))
    if payload.get("status") == "CONTINUATION_REQUIRED":
        print("CONTINUATION: Save & Run All again with identical notebook, inputs, account and T4x2.")
        print("Do not change source SHA, envelope, attempt or durable locator template.")
    elif payload.get("status") == "PASS":
        failed_pub = [cid for cid, row in payload.get("children", {}).items() if row.get("publication_status") != "PASS"]
        if failed_pub:
            print("SCIENCE COMPLETE; public-evidence publication repair is required for:", failed_pub)
        else:
            print("SCIENCE + PUBLIC EVIDENCE: PASS")
    else:
        raise RuntimeError(f"Secondary envelope terminal status is not acceptable: {payload.get('status')}")

elif PHASE == BACKFILL_PHASE:
    manifest, class_map, image_root = find_final_v1()
    os.environ["CROPCOP_MANIFEST"] = str(manifest)
    os.environ["CROPCOP_CLASS_MAP"] = str(class_map)
    os.environ["CROPCOP_IMAGE_ROOT"] = str(image_root)
    os.environ["CROPCOP_OUTPUT_ROOT"] = f"/kaggle/working/cropcop-principal-validation-{BACKFILL_SEED.lower()}"
    os.environ["CROPCOP_VALIDATION_BACKFILL"] = BACKFILL_SEED
    subprocess.run([
        sys.executable, str(repo / "journal_extension/kaggle/run_principal_validation_backfill.py")
    ], cwd=repo, check=True)
else:
    raise RuntimeError(f"Unsupported PHASE={PHASE}")
